# EPL Standings ETL

End-to-end pipeline: API-Football REST API -> Python (requests, pandas) -> SQL Server staging -> warehouse upserts via stored procedures -> Power BI.

Built as a guided training project (Rows & Columns data analytics programme), reconstructed from the project conversation. Final local version may differ.


## 1. Setup and configuration

Install dependencies and load settings. The API key and database details live in a `.env` file, so no secret appears in the notebook.


In [ ]:
# %pip install requests pandas sqlalchemy pyodbc python-dotenv

import json
import requests
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY  = os.environ["APIFOOTBALL_KEY"]
BASE_URL = os.environ.get("APIFOOTBALL_BASE_URL", "https://v3.football.api-sports.io")
LEAGUE   = 39      # Premier League
SEASON   = 2024    # 2024-25 season

# ---- SQL Server settings ----
# Blank user/password = Windows Authentication (no password stored anywhere)
server   = os.environ.get("MSSQL_SERVER", r".\SQLEXPRESS")
database = "PremierLeagueDB"
driver   = "ODBC Driver 17 for SQL Server"



## 2. Extract

Send one GET request per endpoint and **save the reply exactly as received** (raw JSON).


In [ ]:
headers = {"x-apisports-key": API_KEY}
params  = {"league": LEAGUE, "season": SEASON}

def extract(endpoint):
    """Send a GET request to one endpoint and return the raw reply text."""
    response = requests.get(f"{BASE_URL}/{endpoint}", headers=headers, params=params)
    print(f"/{endpoint}  HTTP status:", response.status_code,
          "| characters:", len(response.text))
    return response.text

standings_raw_json = extract("standings")
teams_raw_json     = extract("teams")

print(standings_raw_json[:200])
print(teams_raw_json[:200])


## 3. Parse and validate

HTTP 200 can still carry API errors, so check both replies before using them.


In [ ]:
standings_data = json.loads(standings_raw_json)
teams_data     = json.loads(teams_raw_json)

# HTTP 200 can still carry errors, so check both replies
for name, d in [("standings", standings_data), ("teams", teams_data)]:
    if d["errors"]:
        raise Exception(f"{name} returned errors: {d['errors']}")

standings_list = standings_data["response"][0]["league"]["standings"][0]
teams_list     = teams_data["response"]

print("Standings rows:", len(standings_list))
print("Teams rows:", len(teams_list))


## 4. Transform

Flatten the nested JSON into tidy DataFrames matching the warehouse tables.


In [ ]:
df_standings = pd.json_normalize(standings_list)[[
    "rank", "team.id", "team.name", "points", "goalsDiff", "form",
    "all.played", "all.win", "all.draw", "all.lose",
    "all.goals.for", "all.goals.against"
]].copy()

df_standings.columns = [
    "rank", "team_id", "team_name", "points", "goals_diff", "form",
    "played", "won", "drawn", "lost", "goals_for", "goals_against"
]

df_standings.insert(0, "season", SEASON)   # part of the composite key

print(df_standings.shape)
print(df_standings.head(3))


In [ ]:
# Teams -> df_teams, one row per team (columns match dbo.dim_team).
# NOTE: reconstructed - the original cell was not fully visible in the recovered
# conversation; check it against your final notebook.
df_teams = pd.json_normalize(teams_list)[[
    "team.id", "team.name", "team.code", "team.country", "team.founded",
    "venue.name", "venue.city", "venue.capacity"
]].copy()

df_teams.columns = [
    "team_id", "team_name", "team_code", "country", "founded",
    "venue_name", "venue_city", "venue_capacity"
]

print(df_teams.shape)
print(df_teams.head(3))


## 5. Database and tables

Create the database (run once), connect, then create the warehouse and staging tables.


In [ ]:
master_url = (f"mssql+pyodbc://@{server}/master"
              f"?driver={driver.replace(' ', '+')}&trusted_connection=yes")
master_engine = create_engine(master_url, isolation_level="AUTOCOMMIT")

drop_db_sql = """
IF DB_ID('PremierLeagueDB') IS NOT NULL
BEGIN
    ALTER DATABASE PremierLeagueDB SET SINGLE_USER WITH ROLLBACK IMMEDIATE;
    DROP DATABASE PremierLeagueDB;
END
"""
create_db_sql = "CREATE DATABASE PremierLeagueDB"

with master_engine.connect() as conn:
    conn.execute(text(drop_db_sql))
    conn.execute(text(create_db_sql))
    print("Database created:", conn.execute(text(
        "SELECT name FROM sys.databases WHERE name = 'PremierLeagueDB'")).scalar())


In [ ]:
db_url = (
    f"mssql+pyodbc://@{server}/{database}"
    f"?driver={driver.replace(' ', '+')}&trusted_connection=yes"
)

engine = create_engine(db_url)

with engine.connect() as conn:
    print("Connected to:", conn.execute(text("SELECT DB_NAME()")).scalar())


In [ ]:
create_tables_sql = """
-- ======================================================
-- DIMENSION TABLE: dim_team
-- Describes WHO the team is. One row per team, regardless of season.
-- ======================================================
CREATE TABLE dbo.dim_team (
    team_id        INT            NOT NULL PRIMARY KEY,  -- unique id from the API; the link to standings
    team_name      NVARCHAR(100)  NOT NULL,
    team_code      NVARCHAR(10)   NULL,                  -- NULL allowed: API may not supply every detail
    country        NVARCHAR(50)   NULL,
    founded        INT            NULL,
    venue_name     NVARCHAR(150)  NULL,
    venue_city     NVARCHAR(100)  NULL,
    venue_capacity INT            NULL
);

-- ======================================================
-- STANDINGS TABLE: pl_standings
-- Holds the NUMBERS. One row per team per season.
-- ======================================================
CREATE TABLE dbo.pl_standings (
    season         INT            NOT NULL,
    team_id        INT            NOT NULL,
    rank           INT            NOT NULL,
    team_name      NVARCHAR(100)  NOT NULL,
    points         INT            NOT NULL,
    goals_diff     INT            NOT NULL,
    form           NVARCHAR(20)   NULL,                  -- can be empty early in a season
    played         INT            NOT NULL,
    won            INT            NOT NULL,
    drawn          INT            NOT NULL,
    lost           INT            NOT NULL,

    goals_for      INT            NOT NULL,
    goals_against  INT            NOT NULL,

    -- COMPOSITE PRIMARY KEY: a team can appear only once per season,
    -- so the database itself blocks duplicates
    CONSTRAINT PK_pl_standings PRIMARY KEY (season, team_id),

    -- FOREIGN KEY: every standings row must point to a real team in dim_team
    -- (this is why teams must be loaded BEFORE standings)
    CONSTRAINT FK_standings_team FOREIGN KEY (team_id) REFERENCES dbo.dim_team (team_id)
);

-- ======================================================
-- STAGING TABLES: temporary landing area for each new batch
-- Same columns as the warehouse tables, but no keys.
-- WHERE 1 = 0 copies the structure only, with no rows.
-- ======================================================
SELECT * INTO dbo.stg_dim_team     FROM dbo.dim_team     WHERE 1 = 0;
SELECT * INTO dbo.stg_pl_standings FROM dbo.pl_standings WHERE 1 = 0;
"""

# Python only carries the SQL to the server; engine.begin() runs it as one transaction
with engine.begin() as conn:
    conn.execute(text(create_tables_sql))
print("Tables created")


## 6. Upsert stored procedures

Staging holds each new batch; the procedures merge it into the warehouse - UPDATE what exists, INSERT what is new, in one transaction.


In [ ]:
create_staging_sql = """
IF OBJECT_ID('dbo.stg_pl_standings', 'U') IS NULL
    SELECT * INTO dbo.stg_pl_standings
    FROM dbo.pl_standings
    WHERE 1 = 0
"""

create_proc_sql = """
CREATE OR ALTER PROCEDURE dbo.usp_upsert_pl_standings
AS
BEGIN
    SET NOCOUNT ON;
    SET XACT_ABORT ON;

    BEGIN TRANSACTION;

    -- 1) UPDATE rows that already exist (same season + team_id)
    UPDATE t
    SET t.rank          = s.rank,
        t.team_name     = s.team_name,
        t.points        = s.points,
        t.goals_diff    = s.goals_diff,
        t.form          = s.form,
        t.played        = s.played,
        t.won           = s.won,
        t.drawn         = s.drawn,
        t.lost          = s.lost,
        t.goals_for     = s.goals_for,
        t.goals_against = s.goals_against
    FROM dbo.pl_standings AS t
    JOIN dbo.stg_pl_standings AS s
      ON t.season = s.season AND t.team_id = s.team_id;


    -- 2) INSERT rows that are new
    INSERT INTO dbo.pl_standings
        (season, team_id, rank, team_name, points, goals_diff, form,
         played, won, drawn, lost, goals_for, goals_against)
    SELECT s.season, s.team_id, s.rank, s.team_name, s.points, s.goals_diff, s.form,
           s.played, s.won, s.drawn, s.lost, s.goals_for, s.goals_against
    FROM dbo.stg_pl_standings AS s
    WHERE NOT EXISTS (
        SELECT 1 FROM dbo.pl_standings AS t
        WHERE t.season = s.season AND t.team_id = s.team_id
    );

    COMMIT TRANSACTION;
END
"""

with engine.begin() as conn:
    conn.execute(text(create_staging_sql))
    conn.execute(text(create_proc_sql))

with engine.connect() as conn:
    proc = conn.execute(text(
        "SELECT name FROM sys.procedures WHERE name = 'usp_upsert_pl_standings'"
    )).scalar()
    stg_cols = conn.execute(text(
        "SELECT COUNT(*) FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_NAME = 'stg_pl_standings'"
    )).scalar()
    print("Procedure found:", proc)
    print("Staging columns:", stg_cols)


In [ ]:
# NOTE: reconstructed - follows the same pattern as usp_upsert_pl_standings;
# the original dim_team procedure cell was not fully visible in the recovered conversation.
create_dim_proc_sql = """
CREATE OR ALTER PROCEDURE dbo.usp_upsert_dim_team
AS
BEGIN
    SET NOCOUNT ON;
    SET XACT_ABORT ON;

    BEGIN TRANSACTION;

    UPDATE t
    SET t.team_name     = s.team_name,
        t.team_code     = s.team_code,
        t.country       = s.country,
        t.founded       = s.founded,
        t.venue_name    = s.venue_name,
        t.venue_city    = s.venue_city,
        t.venue_capacity = s.venue_capacity
    FROM dbo.dim_team AS t
    JOIN dbo.stg_dim_team AS s
      ON t.team_id = s.team_id;

    INSERT INTO dbo.dim_team
        (team_id, team_name, team_code, country, founded,
         venue_name, venue_city, venue_capacity)
    SELECT s.team_id, s.team_name, s.team_code, s.country, s.founded,
           s.venue_name, s.venue_city, s.venue_capacity
    FROM dbo.stg_dim_team AS s
    WHERE NOT EXISTS (
        SELECT 1 FROM dbo.dim_team AS t
        WHERE t.team_id = s.team_id
    );

    COMMIT TRANSACTION;
END
"""

with engine.begin() as conn:
    conn.execute(text(create_dim_proc_sql))
print("dim_team procedure created")


## 7. Load

Teams first (standings has a foreign key to dim_team), then standings - one transaction, all or nothing.


In [ ]:
with engine.begin() as conn:   # one transaction: all steps succeed or none do

    # 1) Teams: staging, then upsert into dim_team
    conn.execute(text("TRUNCATE TABLE dbo.stg_dim_team"))
    df_teams.to_sql("stg_dim_team", conn, schema="dbo",
                    if_exists="append", index=False)
    conn.execute(text("EXEC dbo.usp_upsert_dim_team"))

    # 2) Standings: staging, then upsert into pl_standings
    conn.execute(text("TRUNCATE TABLE dbo.stg_pl_standings"))
    df_standings.to_sql("stg_pl_standings", conn, schema="dbo",
                        if_exists="append", index=False)
    conn.execute(text("EXEC dbo.usp_upsert_pl_standings"))

print("Load finished")


## 8. Checks

Query SQL Server directly after every load: row counts per table and duplicate-key check.


In [ ]:
check_counts_sql = """
SELECT 'dim_team'     AS table_name, COUNT(*) AS row_count FROM dbo.dim_team
UNION ALL
SELECT 'pl_standings' AS table_name, COUNT(*) AS row_count FROM dbo.pl_standings
"""

check_dupes_sql = """
SELECT COUNT(*) AS duplicate_keys
FROM (
    SELECT season, team_id
    FROM dbo.pl_standings
    GROUP BY season, team_id
    HAVING COUNT(*) > 1
) AS d
"""

print(pd.read_sql(check_counts_sql, engine))
print(pd.read_sql(check_dupes_sql, engine))
